Using only the most impactful selected Features, Create Model to predict song decade

Using Macro F1 as primary metric, along with weighted F1 and Top 3 accuracy 

Also include macro roc auc , macro pr auc, and log loss

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, LabelBinarizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    roc_auc_score,
    top_k_accuracy_score,
    log_loss
)

from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

Read in the data for decade

In [3]:
X_train = pd.read_csv('data/decade_scaled_X_train.csv')
X_test = pd.read_csv('data/decade_scaled_X_test.csv')
y_train = pd.read_csv('data/decade_y_train.csv')
y_test = pd.read_csv('data/decade_y_test.csv')

Selected Features

In [4]:
selected_features = [
    'bpm',
    'danceability',
    'onset_rate',
    'average_loudness',
    'dynamic_complexity',
    'mfcc_zero_mean',
    'tuning_frequency',
    'tuning_equal_tempered_deviation',
    'mood_happy_prob',
    'mood_aggressive_prob',
    'mood_acoustic',
    'mood_electronic',
    'timbre',
    'voice_instrumental'
]

Keep only selected features

In [5]:
X_train = X_train[selected_features]
X_test = X_test[selected_features]

In [7]:
X_train.columns.tolist()

['bpm',
 'danceability',
 'onset_rate',
 'average_loudness',
 'dynamic_complexity',
 'mfcc_zero_mean',
 'tuning_frequency',
 'tuning_equal_tempered_deviation',
 'mood_happy_prob',
 'mood_aggressive_prob',
 'mood_acoustic',
 'mood_electronic',
 'timbre',
 'voice_instrumental']

In [28]:
X_train.shape

(4223404, 14)

In [8]:
X_test.shape

(1055852, 14)

In [9]:
np.sort(np.unique(y_test))

array([1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020])

Label encode decade

In [29]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


In [11]:
np.unique(y_train)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [12]:
np.unique(y_test)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [30]:
def evaluate_model(y_true, y_prob):

    total_classes = y_prob.shape[1]

    # Derive class predictions from highest probability
    y_pred = y_prob.argmax(axis=1)
    
    # Binarize true labels for multiclass PR-AUC calculations
    lb = LabelBinarizer()
    lb.fit(range(total_classes))  # Ensure all classes are considered
    y_true_binarized = lb.transform(y_true)
    
    # Calculate metrics
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    macro_prauc = average_precision_score(
        y_true_binarized,
        y_prob,
        average='macro'
    )

    try:
        macro_rocauc = roc_auc_score(
            y_true,
            y_prob,
            multi_class='ovr',
            average='macro'
        )
    except ValueError:
        macro_rocauc = np.nan

    top_3_acc = top_k_accuracy_score(y_true, y_prob, k=3, labels=range(total_classes))
    logloss = log_loss(y_true, y_prob, labels=range(total_classes))

    # adjacent decade accuracy, within one decade
    adjacent_accuracy = np.mean(np.abs(y_true - y_pred) <= 1)

    mae_decades = np.mean(np.abs(y_true - y_pred))
    
    # Return as dictionary
    return {
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Macro PR-AUC": macro_prauc,
        "Macro ROC AUC": macro_rocauc,
        "Top-3 Accuracy": top_3_acc,
        "Log-Loss": logloss,
        "Adjacent Accuracy" : adjacent_accuracy,
        "Mean Average Error" : mae_decades
    }

In [ ]:
import pickle
import os

# Compute sample weights dynamically
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

# All models configured to address the class imbalance
models = {
    "Random Forest": RandomForestClassifier(
        max_depth=15,                 # Stops trees from growing infinitely deep
        min_samples_leaf=4,
        class_weight='balanced_subsample', 
        n_jobs=-1, 
        random_state=42
    ),
    
    "LightGBM": lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(set(y_train)) ,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    
    "XGBoost": XGBClassifier(
        objective='multi:softprob', 
        num_class=len(set(y_train)) , 
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ),
    
    "CatBoost": CatBoostClassifier(
        loss_function='MultiClass', 
        auto_class_weights='Balanced',
        random_state=42,
        verbose=50  
    )
}

performance_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    # separate out each model so can view results later
    # added current_model variable to track current model for pickle saving

    if name == "XGBoost":
        xgb_model = model
        xgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)
        current_model = xgb_model

    elif name == "LightGBM":
        lgb_model = model
        lgb_model.fit(
            X_train, y_train,
            sample_weight=sample_weights_train, 
            eval_set=[(X_test, y_test)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)] 
        )
        current_model = lgb_model

    elif name == "CatBoost":
        catboost_model = model
        catboost_model.fit(X_train, y_train)
        current_model = catboost_model

    else:
        rf_model = model
        rf_model.fit(X_train, y_train)
        current_model = rf_model

    # Save Model to Pickle Files

    # Cleans up spaces in name for the filename (i.e., "Random Forest" -> "random_forest_model.pkl")
    clean_name = name.lower().replace(' ', '_')
    filepath = os.path.join("models", f"{clean_name}_decade_model_features_selected.pkl")

    print(f"Saving {name} to {filepath}...")
    with open(filepath, "wb") as file:
        pickle.dump(current_model, file)


    print(f"Evaluating {name} on validation data...")
    if name == "CatBoost":
        y_prob_test = catboost_model.predict_proba(X_test)
    elif name == "LightGBM":
        y_prob_test = lgb_model.predict_proba(X_test)
    elif name == "XGBoost":
        y_prob_test = xgb_model.predict_proba(X_test)
    else:
        y_prob_test = rf_model.predict_proba(X_test)

    
    # Execute the evaluation function
    metrics = evaluate_model(y_test, y_prob_test)
    performance_results[name] = metrics

print("\n" + "="*50)
print("FINAL MODEL COMPARISON RESULTS")
print("="*50)

# Convert results dict to a DataFrame
df_results = pd.DataFrame(performance_results).T
print(df_results.to_string(formatters={c: '{:,.4f}'.format for c in df_results.columns}))


Training Random Forest...
Saving Random Forest to models\random_forest_decade_model_features_selected.pkl...
Evaluating Random Forest on validation data...

FINAL MODEL COMPARISON RESULTS
              Macro F1 Weighted F1 Macro PR-AUC Macro ROC AUC Top-3 Accuracy Log-Loss Adjacent Accuracy Mean Average Error
Random Forest   0.1727      0.2318       0.2013        0.7213         0.6542   1.7908            0.5546             1.8319


My results suggest that Random Forest is usually close, but often misses the exact decade. While, CatBoost gets the exact decade slightly more often but makes larger mistakes when it's wrong.
RF predicts correct decade or neighboring decade 80% of time, while boost models don't. 

In [36]:
catboost_model.classes_

array([0, 1, 2, 3, 4, 5, 6, 7])

In [37]:
lgb_model.classes_

array([0, 1, 2, 3, 4, 5, 6, 7])

In [38]:
xgb_model.classes_

array([0, 1, 2, 3, 4, 5, 6, 7])

In [40]:
rf_model.classes_

array([0, 1, 2, 3, 4, 5, 6, 7])

In [ ]:
# ==================================================
# FINAL MODEL COMPARISON RESULTS
# ==================================================
#               Macro F1 Weighted F1 Macro PR-AUC Macro ROC AUC Top-3 Accuracy Log-Loss Adjacent Accuracy Mean Average Error
# Random Forest   0.1727      0.2318       0.2013        0.7213         0.6542   1.7908            0.5546             1.8319
# LightGBM        0.1818      0.2503       0.2057        0.7308         0.6692   1.7996            0.5733             1.7999
# XGBoost         0.1852      0.2560       0.2086        0.7349         0.6740   1.7842            0.5786             1.7754
# CatBoost        0.1878      0.2548       0.2112        0.7381         0.6818   1.7748            0.5850             1.7385

View confusion matrix for random forest

In [15]:
from sklearn.metrics import confusion_matrix

y_t = y_test
y_p = rf_model.predict(X_test)

matrix = confusion_matrix(y_t, y_p.ravel())

# Define class labels in order
class_labels = [1950,1960,1970,1980,1990,2000,2010,2020] 

matrix_df = pd.DataFrame(
    matrix, 
    index=[f"Actual {label}" for label in class_labels], 
    columns=[f"Predicted {label}" for label in class_labels]
)

matrix_df

,Predicted 1950,Predicted 1960,Predicted 1970,Predicted 1980,Predicted 1990,Predicted 2000,Predicted 2010,Predicted 2020
Actual 1950,15,14,32,24,1318,1633,1943,0
Actual 1960,3,194,232,205,2952,4689,7512,1
Actual 1970,1,67,778,1037,4854,8006,13753,3
Actual 1980,4,49,513,2790,9356,12069,25745,5
Actual 1990,10,153,515,1868,23898,38000,91965,21
Actual 2000,12,137,466,1186,18983,60759,189957,45
Actual 2010,7,88,332,1024,14832,50593,373493,334
Actual 2020,1,7,55,122,1982,7191,77369,650


Save the confusion matrix

In [16]:
matrix_df.to_csv('confusion_matrix/decade_features_selected_rf_confusion_matrix.csv', index=True)

View confusion matrix for catboost

In [17]:
from sklearn.metrics import confusion_matrix

y_t = y_test
y_p = catboost_model.predict(X_test)

matrix = confusion_matrix(y_t, y_p.ravel())

# Define class labels in order
class_labels = [1950,1960,1970,1980,1990,2000,2010,2020] 

matrix_df = pd.DataFrame(
    matrix, 
    index=[f"Actual {label}" for label in class_labels], 
    columns=[f"Predicted {label}" for label in class_labels]
)

matrix_df

,Predicted 1950,Predicted 1960,Predicted 1970,Predicted 1980,Predicted 1990,Predicted 2000,Predicted 2010,Predicted 2020
Actual 1950,3030,780,451,318,181,91,47,81
Actual 1960,4210,6000,2577,1216,503,592,356,334
Actual 1970,5084,5953,8308,4278,1514,1460,817,1085
Actual 1980,7299,5168,7878,16649,4810,3041,2313,3373
Actual 1990,24786,15225,15963,28269,20869,18885,12548,19885
Actual 2000,28748,21404,20827,27800,24610,54654,40570,52932
Actual 2010,27093,25173,22306,32944,26091,60048,91426,155622
Actual 2020,4427,3769,3485,4806,4234,8381,14969,43306


In [18]:
matrix_df.to_csv('confusion_matrix/decade_features_selected_catboost_confusion_matrix.csv', index=True)

Calculate Feature Importance for Random Forest

In [19]:
importances = rf_model.feature_importances_

# Map to feature names and sort
fi_df = pd.DataFrame({
    'feature_names': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(fi_df)

                      feature_names  importance
5                    mfcc_zero_mean    0.103936
7   tuning_equal_tempered_deviation    0.103104
2                        onset_rate    0.099978
3                  average_loudness    0.099750
1                      danceability    0.099390
4                dynamic_complexity    0.097494
0                               bpm    0.094710
8                   mood_happy_prob    0.093206
9              mood_aggressive_prob    0.089681
6                  tuning_frequency    0.079966
11                  mood_electronic    0.010655
12                           timbre    0.010339
13               voice_instrumental    0.009823
10                    mood_acoustic    0.007968


In [ ]:
# feature_names  importance
# 5                    mfcc_zero_mean    0.103936
# 7   tuning_equal_tempered_deviation    0.103104
# 2                        onset_rate    0.099978
# 3                  average_loudness    0.099750
# 1                      danceability    0.099390
# 4                dynamic_complexity    0.097494
# 0                               bpm    0.094710
# 8                   mood_happy_prob    0.093206
# 9              mood_aggressive_prob    0.089681
# 6                  tuning_frequency    0.079966
# 11                  mood_electronic    0.010655
# 12                           timbre    0.010339
# 13               voice_instrumental    0.009823
# 10                    mood_acoustic    0.007968

Calculate feature importance for catboost

In [20]:
importances = catboost_model.get_feature_importance()

# Map to feature names and sort
fi_df = pd.DataFrame({
    'feature_names': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(fi_df)

                      feature_names  importance
6                  tuning_frequency   15.966962
5                    mfcc_zero_mean   13.406183
7   tuning_equal_tempered_deviation   10.379058
2                        onset_rate    9.464923
1                      danceability    8.161373
0                               bpm    7.589721
3                  average_loudness    7.481296
4                dynamic_complexity    6.238963
11                  mood_electronic    4.598862
9              mood_aggressive_prob    4.560275
12                           timbre    3.400752
8                   mood_happy_prob    3.281059
13               voice_instrumental    2.987289
10                    mood_acoustic    2.483285


In [ ]:
# feature_names  importance
# 6                  tuning_frequency   15.966962
# 5                    mfcc_zero_mean   13.406183
# 7   tuning_equal_tempered_deviation   10.379058
# 2                        onset_rate    9.464923
# 1                      danceability    8.161373
# 0                               bpm    7.589721
# 3                  average_loudness    7.481296
# 4                dynamic_complexity    6.238963
# 11                  mood_electronic    4.598862
# 9              mood_aggressive_prob    4.560275
# 12                           timbre    3.400752
# 8                   mood_happy_prob    3.281059
# 13               voice_instrumental    2.987289
# 10                    mood_acoustic    2.483285